# 3. Advanced Workflows and Reproduction

Use this notebook for **robustness checks, projection diagnostics, invariant internals,
and reproducibility** after you already understand the ordinary pipeline.

Application-specific examples are kept in their application notebooks instead of being
repeated here.  In particular:

- Hamiltonian, exceptional-surface, Berry-field, parameter-scan, and material examples
  belong in **Physics Applications**;
- graph-family, Petersen-minor, planarity, dataset, and exact-pattern examples belong
  in **Mathematics Applications**;
- step-by-step graph cleanup belongs in **Core Workflows**.

The advanced notebook therefore focuses only on questions such as:

- Does the extracted graph depend on a delicate numerical parameter?
- Why was a particular planar projection or PD code selected?
- What happens inside the Yamada state expansion?
- Does a geometric deformation change diagram complexity while leaving the invariant stable?

> **Runtime note:** this notebook defaults to `RUN_MODE="quick"`. The
> extraction-sensitivity example uses a smaller exploratory grid in that mode.
> Select `RUN_MODE="paper"` explicitly to restore `dimension=300`; run that
> high-resolution calculation on suitable compute resources. The remaining
> diagnostics work directly with embedded graphs and are substantially lighter.
> The sensitivity section needs the `nodal` and `viz` extras; in a source checkout
> run `uv sync --extra nodal --extra viz --extra notebook`.

## Set up the advanced diagnostics

In [ ]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

candidate = Path.cwd().resolve()
while not (candidate / "src" / "knotted_graph").exists() and candidate != candidate.parent:
    candidate = candidate.parent

if (candidate / "src" / "knotted_graph").exists():
    PROJECT_ROOT = candidate
    SRC_ROOT = PROJECT_ROOT / "src"
    if str(SRC_ROOT) not in sys.path:
        sys.path.insert(0, str(SRC_ROOT))
    installation_mode = f"source checkout: {PROJECT_ROOT}"
elif importlib.util.find_spec("knotted_graph") is not None:
    PROJECT_ROOT = None
    installation_mode = "installed package"
else:
    raise RuntimeError(
        "KnottedGraph was not found. Run this notebook inside a source checkout "
        "or install the current package first."
    )

missing_api = [
    name
    for name in ("knotted_graph.core", "knotted_graph.projection")
    if importlib.util.find_spec(name) is None
]
if missing_api:
    raise RuntimeError(
        "The installed knotted_graph package is missing the current API modules: "
        f"{', '.join(missing_api)}. Install the Latest_Workplace source branch."
    )
import knotted_graph
installation_mode += f" (version {knotted_graph.__version__})"

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

print(f"KnottedGraph mode = {installation_mode}")
for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


In [ ]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

import knotted_graph
from knotted_graph.invariants.yamada.native import (
    native_available,
    native_import_error,
)

print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=4.0, y=4.0, z=3.0))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")

def plot_projection_diagram(
    projection,
    *,
    title=None,
    annotate=False,
    figsize=(5.4, 4.6),
    edge_color=BLUE,
    vertex_color=RED,
    line_width=2.4,
    vertex_size=64,
    gap_fraction=0.015,
):
    """Draw blue edges, red vertices, and explicit over/under crossing gaps."""
    fig, ax = plt.subplots(figsize=figsize)

    arcs_by_id = {
        arc.id: arc
        for arc in projection.arcs
    }

    all_xy = []

    # Draw the complete projected diagram first.
    for arc in projection.arcs:
        x, y = arc.line.xy
        ax.plot(
            x,
            y,
            color=edge_color,
            linewidth=line_width,
            solid_capstyle="round",
            zorder=1,
        )
        all_xy.extend(zip(x, y))

        if annotate:
            midpoint = arc.line.interpolate(
                0.5,
                normalized=True,
            )
            ax.text(
                midpoint.x,
                midpoint.y,
                f"a{arc.id}",
                fontsize=8,
                color=edge_color,
                zorder=7,
            )

    if all_xy:
        xy = np.asarray(
            all_xy,
            dtype=float,
        )
        diagram_span = max(
            float(np.ptp(xy[:, 0])),
            float(np.ptp(xy[:, 1])),
            1.0,
        )
    else:
        diagram_span = 1.0

    gap_radius = (
        gap_fraction
        * diagram_span
    )

    for crossing in projection.crossings:
        try:
            ordered_arcs = list(
                crossing.ccw_ordered_arcs
            )
        except Exception:
            ordered_arcs = []

        if len(ordered_arcs) != 4:
            continue

        cx = crossing.point.x
        cy = crossing.point.y

        # Temporarily erase both strands around the crossing.
        ax.add_patch(
            plt.Circle(
                (cx, cy),
                gap_radius,
                facecolor="white",
                edgecolor="none",
                zorder=4,
            )
        )

        # In KnottedGraph's crossing order, entries 0 and 2 form
        # the over-strand. Redraw those two half-arcs continuously.
        for arc_id in (
            ordered_arcs[0],
            ordered_arcs[2],
        ):
            arc = arcs_by_id.get(
                arc_id
            )

            if (
                arc is None
                or arc.line.length <= 0
            ):
                continue

            probe_distance = min(
                2.4 * gap_radius,
                0.45 * arc.line.length,
            )

            if (
                arc.start_type == "x"
                and arc.start_id
                == crossing.id
            ):
                outside_point = (
                    arc.line.interpolate(
                        probe_distance
                    )
                )

            elif (
                arc.end_type == "x"
                and arc.end_id
                == crossing.id
            ):
                outside_point = (
                    arc.line.interpolate(
                        max(
                            arc.line.length
                            - probe_distance,
                            0.0,
                        )
                    )
                )

            else:
                continue

            ax.plot(
                [cx, outside_point.x],
                [cy, outside_point.y],
                color=edge_color,
                linewidth=line_width,
                solid_capstyle="round",
                zorder=5,
            )

        if annotate:
            ax.annotate(
                f"x{crossing.id}",
                (cx, cy),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
                color="black",
                zorder=8,
            )

    # Rigid graph vertices are always red.
    for vertex in projection.vertices:
        ax.scatter(
            *vertex.point.xy,
            color=vertex_color,
            s=vertex_size,
            zorder=6,
        )

        if annotate:
            ax.annotate(
                f"v{vertex.id}",
                vertex.point.xy,
                xytext=(5, -10),
                textcoords="offset points",
                fontsize=8,
                color=vertex_color,
                zorder=8,
            )

    ax.set_aspect("equal")
    ax.axis("off")

    if title:
        ax.set_title(title)

    plt.show()
    return fig, ax

from plotly.subplots import make_subplots

from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    pq_torus_knot_bloch_vector,
)


def add_surface_trace(
    fig,
    surface,
    *,
    row=1,
    col=1,
    opacity=0.58,
    color=BLUE,
):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points

    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=color,
            opacity=opacity,
            showscale=False,
        ),
        row=row,
        col=col,
    )


def add_points_trace(
    fig,
    points,
    *,
    row=1,
    col=1,
    size=2.5,
    color=BLUE,
):
    points = np.asarray(points)

    fig.add_trace(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(
                size=size,
                color=color,
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )


def style_plotly_scenes(
    fig,
    scene_count,
    *,
    width=980,
    height=660,
):
    for index in range(
        1,
        scene_count + 1,
    ):
        scene_name = (
            "scene"
            if index == 1
            else f"scene{index}"
        )

        fig.update_layout(
            **{
                scene_name: dict(
                    xaxis=axis_style(),
                    yaxis=axis_style(),
                    zaxis=axis_style(),
                    aspectmode="data",
                    camera=CAMERA,
                )
            }
        )

    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(
            l=0,
            r=0,
            t=0,
            b=0,
        ),
        showlegend=False,
    )

    return fig


print("advanced diagnostic helpers ready")


## 3.1 Test extraction robustness: torus $(2,4)$

A topology result should not depend on an arbitrary extraction choice without
being noticed.

The next example compares two thickness parameters. If one setting fails to
produce a usable graph, the failure is reported explicitly so you can inspect the
surface and skeleton rather than assigning an invariant to a questionable graph.

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

For this torus $(2,4)$ sensitivity test,

$$
f_{2,4}(z,w)=z^2-w^4,
\qquad
c=0.7,\quad m=2.
$$

In [ ]:
RUN_MODE = "quick"  # change explicitly to "paper" for high-resolution work
if RUN_MODE not in {"quick", "paper"}:
    raise ValueError("RUN_MODE must be 'quick' or 'paper'.")
SENSITIVITY_DIMENSION = 96 if RUN_MODE == "quick" else 300
print("run mode / grid =", RUN_MODE, SENSITIVITY_DIMENSION)
print("grid points per model =", f"{SENSITIVITY_DIMENSION ** 3:,}")

appendix_torus_records = []
for gamma in (0.06, 0.2):
    ske_t24 = NodalSkeleton(
        pq_torus_knot_bloch_vector(
            2, 4, gamma, k_symbols=(kx, ky, kz), c=0.7, m=2.0
        ),
        k_symbols=(kx, ky, kz),
        dimension=SENSITIVITY_DIMENSION,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_t24 = ske_t24.exceptional_surface_pv
    points_t24 = ske_t24.skeleton_coords
    graph_t24 = None
    graph_error = None
    try:
        graph_t24 = ske_t24.skeleton_graph(simplify=True, smooth_epsilon=2)
    except Exception as exc:
        graph_error = f"{type(exc).__name__}: {exc}"
    appendix_torus_records.append(
        (gamma, surface_t24, points_t24, graph_t24, graph_error)
    )
    print(f"gamma = {gamma}")
    print("  surface_points_cells =", (surface_t24.n_points, surface_t24.n_cells))
    print("  skeleton_points =", points_t24.shape)
    if graph_t24 is None:
        print("  graph_stage =", graph_error)
    else:
        print(
            "  graph_nodes_edges =",
            (graph_t24.number_of_nodes(), graph_t24.number_of_edges()),
        )


In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    horizontal_spacing=0.02,
)
for col, (gamma, surface_t24, points_t24, graph_t24, graph_error) in enumerate(
    appendix_torus_records, start=1
):
    add_surface_trace(fig, surface_t24, row=1, col=col, opacity=0.42)
    add_points_trace(
        fig,
        points_t24,
        row=1,
        col=col,
        size=2.3,
        color=RED if graph_t24 is None else BLUE,
    )
style_plotly_scenes(fig, 2, width=980, height=520).show()


For the parameter value that produces a graph, inspect the embedding directly
before using it in any topological interpretation.


In [ ]:
gamma, surface_t24, points_t24, graph_t24, graph_error = appendix_torus_records[1]
fig = plot_graph_kg(graph_t24)
fig.show()


## 3.2 Diagnose a difficult projection

This section constructs a small embedded graph directly so you can inspect
projection behavior without depending on a domain-specific input adapter.

Use these cells when you want to understand why a particular PD code was produced
or why different viewing directions give diagrams with different crossing counts.

In [ ]:
def diagnostic_k4_spine(samples=90, amplitude=0.75):
    vertices = {
        "a": np.array([-1.15, -0.78, -0.38]),
        "b": np.array([1.18, -0.64, 0.30]),
        "c": np.array([0.86, 0.95, -0.26]),
        "d": np.array([-0.88, 0.84, 0.52]),
    }
    edge_specs = [
        ("a", "b", "ab", np.array([0.00, 0.90, 0.70]), 0.0),
        ("a", "c", "ac", np.array([0.35, -0.15, 1.00]), 1.1),
        ("a", "d", "ad", np.array([0.95, 0.15, -0.25]), 2.2),
        ("b", "c", "bc", np.array([-0.90, 0.25, 0.35]), 0.7),
        ("b", "d", "bd", np.array([-0.20, 1.00, -0.60]), 1.7),
        ("c", "d", "cd", np.array([0.10, -0.90, -0.85]), 2.8),
    ]

    s = np.linspace(0.0, 1.0, samples)
    graph = nx.MultiGraph()
    for vertex_id, pos in vertices.items():
        graph.add_node(vertex_id, pos=pos.copy())

    for u, v, key, bend, phase in edge_specs:
        start = vertices[u]
        end = vertices[v]
        chord = end - start
        bend = bend / np.linalg.norm(bend)
        side = np.cross(chord, bend)
        side = side / np.linalg.norm(side)
        envelope = np.sin(np.pi * s)
        pts = (1 - s)[:, None] * start + s[:, None] * end
        pts += amplitude * envelope[:, None] * (
            np.cos(phase + np.pi * s)[:, None] * bend
            + 0.6 * np.sin(2 * np.pi * s + phase)[:, None] * side
        )
        pts[0] = start
        pts[-1] = end
        graph.add_edge(u, v, key=key, pts=pts)

    graph.graph.update(
        graph_id="advanced_diagnostic_k4",
        input_kind="internal_diagnostic_geometry",
        is_closed=True,
    )
    return graph


diagnostic_graph = diagnostic_k4_spine()


### Inspect how the PD code is assembled

Sample several projections, choose a candidate, and print the diagram objects that
enter the PD code.

This is especially useful when a projection looks visually ambiguous or when you
need to compare two runs at the diagram level.

The over-strand remains continuous and the under-strand is displayed with a small gap.
Optional labels identify the diagram objects, but the graph style remains **blue edges
and red vertices**.

In [ ]:
diagnostic_projection_candidates = sample_projections(
    diagnostic_graph,
    num_rotation_samples=8,
)
nonzero_candidates = [
    candidate
    for candidate in diagnostic_projection_candidates
    if candidate.num_crossings > 0
]
candidate_pool = nonzero_candidates or diagnostic_projection_candidates
pd_emergence_projection = sorted(
    candidate_pool,
    key=lambda candidate: (
        candidate.num_crossings,
        candidate.rotation_angles,
    ),
)[0]

print(
    "rotation_angles =",
    tuple(round(a, 2) for a in pd_emergence_projection.rotation_angles),
)
print(f"crossings = {pd_emergence_projection.num_crossings}")
print("pd_terms:")
for term in sorted(pd_emergence_projection.pd_code.split(";")):
    print(" ", term)
print(
    "vertices =",
    [(vertex.id, vertex.key) for vertex in pd_emergence_projection.vertices],
)
print(
    "crossings =",
    [
        (
            crossing.id,
            tuple(round(c, 3) for c in tuple(crossing.point.coords)[0]),
        )
        for crossing in pd_emergence_projection.crossings
    ],
)
print(
    "arcs =",
    [
        (arc.id, arc.start_type, arc.start_id, arc.end_type, arc.end_id)
        for arc in pd_emergence_projection.arcs
    ],
)


In [ ]:
plot_projection_diagram(
    pd_emergence_projection,
    title="PD-code projection with explicit over/under crossings",
    annotate=True,
    figsize=(6.0, 5.0),
)


## 3.3 Inspect the Yamada state expansion

For ordinary calculations, use the public
`compute_yamada_polynomial(...)` interface.

The next cells inspect the exact compact states used by the production Yamada
calculation through `Yamada.iter_compact_states()`. This diagnostic therefore
examines the same optimized state representation used by ordinary evaluation,
rather than materializing a separate legacy NetworkX state expansion.

In [ ]:
from knotted_graph.invariants.yamada import Yamada, compute_yamada_from_states

yamada_inspector = Yamada(
    pd_emergence_projection.vertices,
    pd_emergence_projection.crossings,
    pd_emergence_projection.arcs,
)
state_records = list(yamada_inspector.iter_compact_states())
state_graphs = [graph for graph, _ in state_records]
exponents = [exponent for _, exponent in state_records]
state_sum = compute_yamada_from_states(
    state_graphs,
    exponents,
    Y,
    method="negami",
    n_jobs=1,
)
public_result = compute_yamada_polynomial(
    diagnostic_graph,
    Y,
    rotation_angles=pd_emergence_projection.rotation_angles,
    n_jobs=1,
)

print(f"crossings = {pd_emergence_projection.num_crossings}")
print(f"state_count = {len(state_graphs)}")
for index, (state_graph, exponent) in enumerate(
    zip(state_graphs[:6], exponents[:6])
):
    print(
        f"state[{index}] exponent={exponent} "
        f"nodes_edges={(state_graph.n, state_graph.edge_count)}"
    )
print_upsilon("state_sum", state_sum)
print_upsilon("public_call", public_result)
print("same_result =", sp.expand(state_sum - public_result) == 0)


In [ ]:
num_panels = min(3, len(state_graphs))
fig, axes = plt.subplots(1, num_panels, figsize=(3.0 * num_panels, 3.0))
axes = np.atleast_1d(axes)

for axis, state_index in zip(axes, range(num_panels)):
    state_graph = state_graphs[state_index]
    simple_state = nx.Graph()
    simple_state.add_nodes_from(range(state_graph.n))
    for i in range(state_graph.n):
        for j in range(i, state_graph.n):
            if state_graph.rows[i][j]:
                simple_state.add_edge(i, j)
    positions = nx.spring_layout(simple_state, seed=10 + state_index)
    nx.draw_networkx_edges(
        simple_state,
        positions,
        ax=axis,
        edge_color=BLUE,
        width=2.0,
    )
    nx.draw_networkx_nodes(
        simple_state,
        positions,
        ax=axis,
        node_color=RED,
        node_size=90,
    )
    axis.set_aspect("equal")
    axis.axis("off")

plt.show()


### Check the state-level result against the public API

The state-level calculation and the public call should agree when they use the
same selected diagram.

Use this comparison when debugging the invariant calculation itself. For normal
library use, prefer the public result object.


In [ ]:
diagnostic_public_result = compute_yamada_polynomial(
    diagnostic_graph,
    Y,
    rotation_angles=pd_emergence_projection.rotation_angles,
    return_result=True,
    n_jobs=1,
)
print(
    "nodes_edges =",
    (diagnostic_graph.number_of_nodes(), diagnostic_graph.number_of_edges()),
)
print(
    "selected_crossings =",
    diagnostic_public_result.projection.num_crossings,
)
print(f"pd_code = {diagnostic_public_result.projection.pd_code}")
print_upsilon("G_diagnostic", diagnostic_public_result.polynomial)


## 3.4 Test projection robustness under geometric scaling

The next example changes the geometric representative by scaling one coordinate
and then samples projections at each scale.

Record both the crossing-count range and the selected invariant. This tells you
how sensitive the **diagram selection** is to the geometry.

An unchanged polynomial across these tests is useful evidence of robustness, but
this numerical experiment is not a proof of ambient isotopy. If the polynomial
changes unexpectedly, inspect the geometry, projection validity, and PD code
before drawing a topological conclusion.

In [ ]:
def scale_embedded_graph_x(source_graph, x_scale):
    scaled = nx.MultiGraph()
    for node, data in source_graph.nodes(data=True):
        attrs = dict(data)
        attrs["pos"] = np.asarray(attrs["pos"], dtype=float) * np.array(
            [x_scale, 1.0, 1.0]
        )
        scaled.add_node(node, **attrs)

    for u, v, key, data in source_graph.edges(keys=True, data=True):
        attrs = dict(data)
        attrs["pts"] = np.asarray(attrs["pts"], dtype=float) * np.array(
            [x_scale, 1.0, 1.0]
        )
        scaled.add_edge(u, v, key=key, **attrs)

    scaled.graph.update(source_graph.graph)
    scaled.graph["x_scale"] = x_scale
    return scaled


projection_diagnostics = []
for x_scale in (0.75, 1.0, 1.25, 1.5):
    scaled_graph = scale_embedded_graph_x(diagnostic_graph, x_scale)
    candidates = sorted(
        sample_projections(scaled_graph, num_rotation_samples=8),
        key=lambda candidate: (
            candidate.num_crossings,
            candidate.rotation_angles,
        ),
    )
    crossing_counts = [candidate.num_crossings for candidate in candidates]
    selected = candidates[0]
    polynomial = compute_yamada_polynomial(
        scaled_graph,
        Y,
        rotation_angles=selected.rotation_angles,
        n_jobs=1,
    )
    projection_diagnostics.append(
        {
            "x_scale": x_scale,
            "min_crossings": min(crossing_counts),
            "median_crossings": float(np.median(crossing_counts)),
            "max_crossings": max(crossing_counts),
            "selected_angles": tuple(
                round(a, 2) for a in selected.rotation_angles
            ),
            "polynomial": sp.expand(polynomial),
        }
    )

for row in projection_diagnostics:
    print(f"x_scale = {row['x_scale']}")
    print(
        "  crossing_range =",
        (
            row["min_crossings"],
            row["median_crossings"],
            row["max_crossings"],
        ),
    )
    print("  selected_angles =", row["selected_angles"])
    print_upsilon("G_scaled", row["polynomial"])


In [ ]:
scales = [row["x_scale"] for row in projection_diagnostics]
fig, ax = plt.subplots(figsize=(5.4, 3.8))
ax.plot(
    scales,
    [row["min_crossings"] for row in projection_diagnostics],
    color=BLUE,
    marker="o",
    label="min",
)
ax.plot(
    scales,
    [row["median_crossings"] for row in projection_diagnostics],
    color="black",
    marker="s",
    label="median",
)
ax.plot(
    scales,
    [row["max_crossings"] for row in projection_diagnostics],
    color=RED,
    marker="^",
    label="max",
)
ax.set_xlabel("x scale")
ax.set_ylabel("projection crossings")
ax.legend(frameon=False)
plt.show()


## 3.5 Reproduction checklist

Use this notebook only after the ordinary workflow already produces a sensible graph
and Yamada result.  Before reporting a topology result, record the numerical choices
that can affect the extracted graph or selected diagram:

1. the input model and its parameters;
2. the sampled momentum-space span and grid resolution;
3. the extraction / skeletonization parameters;
4. the graph-simplification parameters used in **Core Workflows**;
5. the selected projection angles and crossing count;
6. the exact PD code;
7. the Yamada convention and computation settings; and
8. at least one robustness check when the extraction or projection is numerically delicate.

Application examples are intentionally **not repeated here**.  Use the authoritative
application notebook for Hamiltonians, Berry fields, materials, graph families,
Petersen-minor tests, planarity scans, and datasets.

When a diagnostic reveals a problem, return to the stage that produced it:

- extraction or simplification issue → **[2. Core Workflows](02_core_workflows.ipynb)**
- physical model / field / material issue → **[Physics Applications](applications/01_physics_applications.ipynb)**
- graph-family, minor, planarity, or exact-polynomial question → **[Mathematics Applications](applications/02_mathematics_applications.ipynb)**
- protein-domain conversion issue → **[Protein Applications](applications/03_protein_applications.ipynb)**

Return to the **[User Guide Index](00_user_guide.ipynb)** to choose another workflow.